# Trial 01 Results and Figures

Publication figures from `preliminary_results.pkl`, `regression_results.pkl`, and
`classification_results.pkl`. 

Styling uses SciencePlots (`['science', 'ieee']`); install with
`pip install SciencePlots`. Gas colours are Paul Tol's bright scheme extended to
10 hues.

## Setup and Style

In [ ]:
# Imports
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401 -- registers the 3D projection
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# IEEE style via SciencePlots
import scienceplots   # noqa: F401 -- registers styles with matplotlib
plt.style.use(['science', 'ieee', 'no-latex'])

# Column-width helpers (inches; IEEE Sensors / TIM / JSEN conventions)
COL_SINGLE = 3.5      # single-column width
COL_DOUBLE = 7.16     # full text width

def figsize(width='single', aspect=0.75):
    '''Return (w, h) in inches. `width` in {'single','double'}, aspect = h/w.'''
    w = COL_SINGLE if width == 'single' else COL_DOUBLE
    return (w, w * aspect)

# Global figure defaults
# no savefig.bbox='tight': it rescales the saved figure unpredictably.
# use explicit subplots_adjust() so every PDF matches figsize exactly and
# LaTeX \includegraphics[width=\columnwidth]{} needs no font scaling.
plt.rcParams.update({
    'figure.dpi':            120,     # screen preview
    'savefig.dpi':           600,     # camera-ready
    'savefig.format':        'pdf',
    # 'savefig.pad_inches':    0.0,     # no extra whitespace around figure
    # 'axes.titlesize':        8,
    # 'axes.labelsize':        8,
    # 'xtick.labelsize':       7,
    # 'ytick.labelsize':       7,
    # 'legend.fontsize':       7,
    # 'legend.title_fontsize': 7,
    # 'lines.linewidth':       0.9,
    # 'axes.linewidth':        0.6,
    # 'grid.linewidth':        0.4,
    # 'font.family':           'serif',
    # 'font.serif':            ['DejaVu Serif', 'Times New Roman', 'Times'],
    # 'mathtext.fontset':      'dejavuserif',
})

# Paul Tol qualitative palette (10 colours)
TOL10 = [
    '#4477AA',  # blue          (bright)
    '#EE6677',  # red           (bright)
    '#228833',  # green         (bright)
    '#CCBB44',  # yellow        (bright)
    '#66CCEE',  # cyan          (bright)
    '#AA3377',  # purple        (bright)
    '#EE7733',  # orange        (vibrant)
    '#009988',  # teal          (vibrant)
    # '#994455',  # wine          (muted)
    "#000000",
    '#BBBBBB',  # grey          (bright)
]

# Model colours (solid only; no hatching)
COLOR_SVR, COLOR_GB, COLOR_GP = TOL10[0], TOL10[6], TOL10[2]

SHORT  = {'PCA + SVR': 'SVR',  'PCA + GB': 'GB',  'PCA + GP': 'GP',
          'PCA + SVC': 'SVC',  'PCA + GBC': 'GBC', 'PCA + GPC': 'GPC'}
COLORS = {'PCA + SVR': COLOR_SVR, 'PCA + GB':  COLOR_GB, 'PCA + GP':  COLOR_GP,
          'PCA + SVC': COLOR_SVR, 'PCA + GBC': COLOR_GB, 'PCA + GPC': COLOR_GP}
HATCH  = {k: '' for k in COLORS}
LSTYLE = {'PCA + SVR': '-',  'PCA + GB':  '--',  'PCA + GP':  ':',
          'PCA + SVC': '-',  'PCA + GBC': '--',  'PCA + GPC': ':'}

def remap_gas_colors(gas_types):
    return {gas: TOL10[i % len(TOL10)] for i, gas in enumerate(gas_types)}

def panel_label(ax, text, x=-0.15, y=1.02):
    '''Bold (a), (b), (c) panel labels - IEEE convention.
    Works for both 2D and 3D axes.'''
    if hasattr(ax, 'text2D'):
        ax.text2D(x, y, text, transform=ax.transAxes,
                  fontsize=9, fontweight='bold', va='bottom', ha='left')
    else:
        ax.text(x, y, text, transform=ax.transAxes,
                fontsize=9, fontweight='bold', va='bottom', ha='left')

def gas_legend_right(fig, gas_types, gas_colors):
    '''Place the gas legend on the right side of the figure. Call AFTER
    subplots_adjust has reserved right margin (right <= 0.82).'''
    handles = [mpatches.Patch(color=gas_colors[g], label=g) for g in gas_types]
    return fig.legend(handles=handles, title='Gas', ncol=1,
                      loc='center right', bbox_to_anchor=(0.95, 0.5),
                      handlelength=1.0, handletextpad=0.4,
                      frameon=True, edgecolor='0.8', fancybox=False)

def gas_legend_under(fig, gas_types, gas_colors):
    '''Place the gas legend on the right side of the figure. Call AFTER
    subplots_adjust has reserved right margin (right <= 0.82).'''
    handles = [mpatches.Patch(color=gas_colors[g], label=g) for g in gas_types]
    return fig.legend(handles=handles, title='Gas', ncol=5,
                      loc='lower center', bbox_to_anchor=(0.5, -0.3),
                      handlelength=1.0, handletextpad=0.4,
                      frameon=True, edgecolor='0.8', fancybox=False)

# Paths
PRELIM_PATH = 'preliminary_results.pkl'
REG_PATH    = 'regression_results.pkl'
CLF_PATH    = 'classification_results.pkl'
OUT_DIR     = '01_results'
os.makedirs(OUT_DIR, exist_ok=True)

print('Setup complete. Fixed-dimension mode: no bbox=tight, explicit subplots_adjust.')


## Load Results

In [ ]:
prelim_available = os.path.exists(PRELIM_PATH)
reg_available    = os.path.exists(REG_PATH)
clf_available    = os.path.exists(CLF_PATH)

# Preliminary
if prelim_available:
    with open(PRELIM_PATH, 'rb') as f:
        prelim = pickle.load(f)
    # The upstream pickle uses em-dashes in its keys; match them.
    SKEY_A = next((k for k in prelim if isinstance(k, str) and 'Sensor A' in k), None)
    SKEY_B = next((k for k in prelim if isinstance(k, str) and 'Sensor B' in k), None)
    assert SKEY_A and SKEY_B, 'Sensor keys not found in preliminary pickle'

    NOISE_LEVEL = prelim['noise_level']
    proj        = prelim['projections']
    y_gas       = proj['y_gas']
    GAS_TYPES_P = proj['gas_types']
    GAS_COLORS  = remap_gas_colors(GAS_TYPES_P)   # <-- override upstream palette
    print(f'preliminary: noise={NOISE_LEVEL:.4f}, gases={len(GAS_TYPES_P)}, '
          f'palette remapped to Tol10')
else:
    print('preliminary_results.pkl missing - preliminary figures will be skipped')

# Regression
if reg_available:
    with open(REG_PATH, 'rb') as f:
        reg_bundle = pickle.load(f)
    cv_results      = reg_bundle['cv_results']
    df_reg          = reg_bundle['regression_df']
    MODEL_NAMES_REG = reg_bundle['metadata']['model_names']
    SENSOR_NAMES    = list(cv_results.keys())
    GAS_TYPES       = reg_bundle.get('gas_types',
                          sorted(list(next(iter(cv_results.values())).keys())))
    print(f'regression: sensors={len(SENSOR_NAMES)}, gases={len(GAS_TYPES)}, '
          f'models={MODEL_NAMES_REG}')
else:
    print('regression_results.pkl missing - regression figures will be skipped')

# Classification + LOCO
if clf_available:
    with open(CLF_PATH, 'rb') as f:
        clf_bundle = pickle.load(f)
    clf_results      = clf_bundle['clf_results']
    le               = clf_bundle['label_encoder']
    MODEL_NAMES_CLF  = clf_bundle['metadata']['model_names']
    SENSOR_NAMES_CLF = list(clf_results.keys())
    if not reg_available:
        SENSOR_NAMES = SENSOR_NAMES_CLF
    loco_available = 'clf_results_loco' in clf_bundle
    if loco_available:
        loco_results    = clf_bundle['clf_results_loco']
        loco_summary_df = clf_bundle['loco_summary_df']
    print(f'classification: sensors={len(SENSOR_NAMES_CLF)}, '
          f'models={MODEL_NAMES_CLF}, LOCO={loco_available}')
else:
    loco_available = False
    print('classification_results.pkl missing - classification figures will be skipped')

def sensor_short_label(name):
    '''"Sensor A - GO/Nafion (S11)" -> "Sensor A", works for em-dash and hyphen.'''
    for sep in (' - ', ' -- ', chr(8212), chr(8211), '-'):
        if sep in name:
            return name.split(sep)[0].strip()
    return name


## Preliminary: PCA Explained Variance

In [ ]:
if prelim_available:
    evr_a    = prelim[SKEY_A]['evr_per_component']
    evr_b    = prelim[SKEY_B]['evr_per_component']
    cumvar_a = prelim[SKEY_A]['cumulative_evr']
    cumvar_b = prelim[SKEY_B]['cumulative_evr']
    n_comp_a = prelim[SKEY_A]['n_components']
    n_comp_b = prelim[SKEY_B]['n_components']
    letters  = [char for char in 'abcdef']

    for evr, cumvar, n_comp, skey, title in zip(
            [evr_a, evr_b],
            [cumvar_a, cumvar_b],
            [n_comp_a, n_comp_b],
            [SKEY_A, SKEY_B],
            ['Sensor A', 'Sensor B']):

        fig, ax = plt.subplots()

        xs = np.arange(1, len(evr) + 1)
        ax.bar(xs, evr, color=COLOR_SVR, alpha=0.8, label='Individual',
               edgecolor='black', linewidth=0.4)
        print(evr)
        ax2 = ax.twinx()
        ax2.plot(xs, cumvar, color=COLOR_GB, lw=1.1, marker='o', ms=2.5,
                 label='Cumulative')
        ax2.axhline(0.95, color='0.3', ls='--', lw=0.7, label='95 %')
        ax2.axvline(n_comp, color='0.5', ls=':', lw=0.7)
        ax2.annotate(f'n={n_comp}', xy=(n_comp + 0.4, 0.88),
                     fontsize=6, color='0.3', va='bottom')
        ax2.set_ylim(0, 1.05)
        ax.set_ylim(0, 0.65)
        ax.set_xlabel('Principal Component')
        ax.set_ylabel('Explained Variance Ratio')
        ax2.set_ylabel('Cumulative EVR')
        ax.set_title(title)
        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax.legend(h1 + h2, l1 + l2, loc='lower right', frameon=True)

        sshort = skey.split('\u2014')[0].strip().lower().replace(' ', '_')
        fname  = f'{OUT_DIR}/fig_evr_{sshort}.pdf'
        plt.savefig(fname)
        print(f'Saved {fname}')
        plt.show()

## Preliminary: PCA 2D

In [ ]:
if prelim_available:
    sensor_pairs = [(SKEY_A, 'sensor_a', 'Sensor A'),
                    (SKEY_B, 'sensor_b', 'Sensor B')]

    for skey, sshort, slabel in sensor_pairs:
        X_pca = proj[skey]['X_pca']

        fig, ax = plt.subplots()
        # fig.subplots_adjust(left=0.14, right=0.75, bottom=0.14, top=0.90)

        for gas in GAS_TYPES_P:
            mask = y_gas == gas
            ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                       c=[GAS_COLORS[gas]], s=5, alpha=0.5,
                       edgecolors='none', rasterized=True)
        ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2')
        ax.set_title(f'{slabel}, PCA')

        # gas_legend_right(fig, GAS_TYPES_P, GAS_COLORS)
        # gas_legend_under(fig, GAS_TYPES_P, GAS_COLORS)

        fname = f'{OUT_DIR}/fig_pca_{sshort}.pdf'
        plt.savefig(fname)
        print(f'Saved {fname}')
        plt.show()


## Preliminary: PCA 3D

In [ ]:
if prelim_available:
    fig, ax = plt.subplots(figsize=(0.5, 0.5))
    ax.axis('off')

    for gas in GAS_TYPES_P:
        ax.scatter([], [], c=[GAS_COLORS[gas]], s=15, marker='s', label=gas)

    ax.legend(title='Gas Type', loc='center', frameon=True,
              ncol=5, markerscale=1.5)

    fname = f'{OUT_DIR}/fig_gas_legend.pdf'
    plt.savefig(fname, bbox_inches='tight')
    print(f'Saved {fname}')
    plt.show()

In [ ]:
if prelim_available:
    lines = []
    for gas, color in GAS_COLORS.items():
        # color may be a hex string or an RGB tuple
        if isinstance(color, str):
            hex_color = color.lstrip('#')
        else:
            hex_color = '{:02x}{:02x}{:02x}'.format(
                int(color[0]*255), int(color[1]*255), int(color[2]*255))
        safe_name = gas.lower().replace(' ', '').replace('-', '')
        lines.append(f'\\definecolor{{gas{safe_name}}}{{HTML}}{{{hex_color.upper()}}}')

    latex_block = '\n'.join(lines)
    print(latex_block)

    # fname = f'{OUT_DIR}/gas_colors.tex'
    # with open(fname, 'w') as f:
    #     f.write(latex_block + '\n')
    # print(f'Saved {fname}')

if prelim_available:
    color_entries = []
    for gas, color in GAS_COLORS.items():
        if isinstance(color, str):
            hex_color = color.lstrip('#').upper()
        else:
            hex_color = '{:02X}{:02X}{:02X}'.format(
                int(color[0]*255), int(color[1]*255), int(color[2]*255))
        safe_name = 'gas' + gas.lower().replace(' ', '').replace('-', '')
        color_entries.append((gas, safe_name, hex_color))

    # Build caption string
    legend_parts = ', '.join(
        f'\\textcolor{{{safe_name}}}{{\\rule{{0.8em}}{{0.8em}}}} {gas}'
        for gas, safe_name, _ in color_entries)
    caption = (
        f'Caption text here. '
        f'Gas types are indicated by colour: {legend_parts}.'
    )

    print(caption)

    # fname = f'{OUT_DIR}/fig_caption.tex'
    # with open(fname, 'w') as f:
    #     f.write(caption + '\n')
    # print(f'Saved {fname}')

In [ ]:
if prelim_available:
    fig = plt.figure(figsize=figsize('double', 0.4))
    fig.subplots_adjust(left=0.02, right=0.80, bottom=0.08, top=0.92, wspace=0.05)
    # fig.subplots_adjust(wspace=-0.5,hspace=0.0)

    for col, skey, slabel, plabel in zip(
            [0, 1],
            [SKEY_A, SKEY_B],
            ['Sensor A', 'Sensor B'],
            ['(a)', '(b)']):
        X_pca = proj[skey]['X_pca']
        ax = fig.add_subplot(1, 2, col + 1, projection='3d')
        for gas in GAS_TYPES_P:
            mask = y_gas == gas
            ax.scatter(X_pca[mask, 0], X_pca[mask, 1], X_pca[mask, 2],
                       c=[GAS_COLORS[gas]], s=4, alpha=0.4,
                       edgecolors='none')
        ax.set_xlabel('PC 1', labelpad=2)
        ax.set_ylabel('PC 2', labelpad=2)
        ax.set_zlabel('PC 3', labelpad=2)
        ax.tick_params(axis='both', labelsize=6)
        ax.set_title(f'{slabel}, PCA 3D')
        ax.set_box_aspect((1,1,1), zoom=0.85)
        ax.view_init(elev=20, azim=-60)
        # panel_label(ax, plabel, x=0.02, y=1.02)

    # gas_legend_right(fig, GAS_TYPES_P, GAS_COLORS)

    fname = f'{OUT_DIR}/fig_pca_3d.pdf'
    plt.savefig(fname)
    print(f'Saved {fname}')
    plt.show()


## Preliminary: LDA 2D

In [ ]:
if prelim_available:
    sensor_pairs = [(SKEY_A, 'sensor_a', 'Sensor A'),
                    (SKEY_B, 'sensor_b', 'Sensor B')]

    for skey, sshort, slabel in sensor_pairs:
        X_lda = proj[skey]['X_lda']

        fig, ax = plt.subplots()
        # fig.subplots_adjust(left=0.14, right=0.75, bottom=0.14, top=0.90)

        for gas in GAS_TYPES_P:
            mask = y_gas == gas
            ax.scatter(X_lda[mask, 0], X_lda[mask, 1],
                       c=[GAS_COLORS[gas]], s=5, alpha=0.5,
                       edgecolors='none', rasterized=True)
        ax.set_xlabel('LD 1'); ax.set_ylabel('LD 2')
        ax.set_title(f'{slabel}, LDA')

        # gas_legend_right(fig, GAS_TYPES_P, GAS_COLORS)
        # gas_legend_under(fig, GAS_TYPES_P, GAS_COLORS)

        fname = f'{OUT_DIR}/fig_lda_{sshort}.pdf'
        plt.savefig(fname)
        print(f'Saved {fname}')
        plt.show()

## Preliminary: LDA 3D

In [ ]:
if prelim_available:
    fig = plt.figure(figsize=figsize('double', 0.4))
    fig.subplots_adjust(left=0.02, right=0.80, bottom=0.08, top=0.92, wspace=0.05)

    for col, skey, slabel, plabel in zip(
            [0, 1],
            [SKEY_A, SKEY_B],
            ['Sensor A', 'Sensor B'],
            ['(a)', '(b)']):
        X_lda = proj[skey]['X_lda']
        ax = fig.add_subplot(1, 2, col + 1, projection='3d')
        for gas in GAS_TYPES_P:
            mask = y_gas == gas
            ax.scatter(X_lda[mask, 0], X_lda[mask, 1], X_lda[mask, 2],
                       c=[GAS_COLORS[gas]], s=4, alpha=0.4,
                       edgecolors='none')
        ax.set_xlabel('LD 1', labelpad=2)
        ax.set_ylabel('LD 2', labelpad=2)
        ax.set_zlabel('LD 3', labelpad=2)
        ax.tick_params(axis='both', labelsize=6)
        ax.set_title(f'{slabel}, LDA 3D')
        ax.set_box_aspect((1, 1, 1), zoom=0.85)
        ax.view_init(elev=20, azim=-60)
        # panel_label(ax, plabel, x=0.02, y=1.02)

    gas_legend_right(fig, GAS_TYPES_P, GAS_COLORS)

    fname = f'{OUT_DIR}/fig_lda_3d.pdf'
    plt.savefig(fname)
    print(f'Saved {fname}')
    plt.show()


## Preliminary: t-SNE Scatter

Perplexity selected by KNN neighbourhood preservation.

In [ ]:
if prelim_available:
    sensor_pairs = [(SKEY_A, 'sensor_a', 'Sensor A'),
                    (SKEY_B, 'sensor_b', 'Sensor B')]

    for skey, sshort, slabel in sensor_pairs:
        X_tsne    = proj[skey]['X_tsne']
        best_perp = prelim[skey]['best_tsne_perplexity']

        fig, ax = plt.subplots()

        for gas in GAS_TYPES_P:
            mask = y_gas == gas
            ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                       c=[GAS_COLORS[gas]], s=5, alpha=0.5,
                       edgecolors='none', rasterized=True)
        ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
        ax.set_title(f'{slabel}, t-SNE')
        
        ax.set_xlim(-125,125)
        ax.set_ylim(-125,125)

        # gas_legend_right(fig, GAS_TYPES_P, GAS_COLORS)
        # gas_legend_under(fig, GAS_TYPES_P, GAS_COLORS)

        fname = f'{OUT_DIR}/fig_tsne_{sshort}.pdf'
        plt.savefig(fname)
        print(f'Saved {fname}')
        plt.show()


## Regression: R² and MAE Grouped Bars

Error bars are 95% CIs from cross-validation; models differ by hue.

In [ ]:
if reg_available:
    n_models      = len(MODEL_NAMES_REG)
    n_gases       = len(GAS_TYPES)
    group_w       = 0.7
    bar_w         = group_w / n_models
    x_centers     = np.arange(n_gases)
    sensor_labels = [sensor_short_label(s) for s in SENSOR_NAMES]
    letters       = [char for char in 'abcdefgh']

    for metric_key, ylabel, ci_key, ylim, fname_stub in [
            ('r2',  r'$R^2$',    'ci_r2',  (0.0, 1.1), 'r2'),
            ('mae', 'MAE (ppm)', 'ci_mae', (0, 130),    'mae'),
    ]:
        for sensor_name, slabel in zip(SENSOR_NAMES, sensor_labels):
            fig, ax = plt.subplots()

            for mi, model_name in enumerate(MODEL_NAMES_REG):
                offsets = x_centers + (mi - (n_models - 1) / 2) * bar_w
                vals, lo_errs, hi_errs = [], [], []
                for gas_type in GAS_TYPES:
                    res = cv_results[sensor_name][gas_type][model_name]
                    v   = res[metric_key]
                    ci  = res[ci_key]
                    vals.append(v)
                    lo_errs.append(v - ci[1])
                    hi_errs.append(ci[2] - v)
                ax.bar(offsets, vals, width=bar_w * 0.9,
                       color=COLORS[model_name],
                       edgecolor='black', linewidth=0.5, zorder=3,
                       label=SHORT[model_name])
                ax.errorbar(offsets, vals, yerr=[lo_errs, hi_errs],
                            fmt='none', color='black', capsize=1.5,
                            lw=0.5, zorder=4)

            ax.set_xticks(x_centers)
            ax.set_xticklabels(GAS_TYPES, rotation=45, ha='right')
            ax.set_title(slabel)
            ax.set_ylabel(ylabel)
            ax.legend(loc='best', title='Model', frameon=True)
            if ylim:
                ax.set_ylim(ylim)

            # ax.text(-0.20, 1.05, f'({letters.pop(0)})', transform=ax.transAxes,
            #         fontsize=8, fontweight='bold', va='top')

            sshort = sensor_name.split('\u2014')[0].strip().lower().replace(' ', '_')
            fname  = f'{OUT_DIR}/fig_{fname_stub}_{sshort}.pdf'
            plt.savefig(fname)
            print(f'Saved {fname}')
            plt.show()

## Regression: True vs Predicted Scatter

One 5×2 grid per sensor and model. Dashed red = ideal y = x; `aspect='equal'`
so that line sits at 45°.

In [ ]:
if reg_available:
    for model_name in MODEL_NAMES_REG:
        ncols = 5
        nrows = (len(GAS_TYPES) + ncols - 1) // ncols

        for sensor_name in SENSOR_NAMES:
            sshort = 'sensor_a' if 'GO/Nafion' in sensor_name else 'sensor_b'
            fig, axes = plt.subplots(nrows, ncols,
                                     figsize=figsize('double', 0.5),
                                     sharex=True, sharey=True)
            fig.subplots_adjust(left=0.06, right=0.97, bottom=0.10,
                                top=0.92, wspace=0.15, hspace=0.05)
            axes_flat = axes.flatten()

            for i, gas_type in enumerate(GAS_TYPES):
                ax  = axes_flat[i]
                res = cv_results[sensor_name][gas_type][model_name]
                yt, yp = res['y_true'], res['y_pred']
                ax.scatter(yt, yp, c=yt, cmap='viridis', alpha=0.4,
                           s=2, edgecolors='none', rasterized=True)
                ax.plot([0, 1250], [0, 1250], color=TOL10[1], ls='--', lw=0.6)
                ax.set_title(gas_type, fontsize=7, pad=2)
                ax.annotate(f"$R^2$={res['r2']:.2f}",
                            xy=(0.04, 0.96), xycoords='axes fraction',
                            va='top', ha='left', fontsize=6,
                            bbox=dict(boxstyle='round,pad=0.15',
                                      fc='white', ec='none', alpha=0.8))
                ax.tick_params(labelsize=6)
                ax.set_aspect('equal', adjustable='box')

            for j, ax in enumerate(axes_flat[:len(GAS_TYPES)]):
                if j % ncols == 0:
                    ax.set_ylabel('Predicted (ppm)', fontsize=7)
                if j >= (nrows - 1) * ncols:
                    ax.set_xlabel('True (ppm)', fontsize=7)

            for j in range(len(GAS_TYPES), len(axes_flat)):
                axes_flat[j].set_visible(False)

            fig.suptitle(f"{sensor_short_label(sensor_name)}, {SHORT[model_name]}")
            fname = f'{OUT_DIR}/fig_scatter_{sshort}_{SHORT[model_name].lower()}.pdf'
            plt.savefig(fname)
            print(f'Saved {fname}')
            plt.show()
            

## Classification (LOCO): Per-fold Accuracy and F1

Each fold holds out 4 consecutive dose ranks. Grey dashed = 0.95 reference.

In [ ]:
if loco_available:
    n_folds     = 5
    fold_labels = [f'F{i+1}\nr{i*4+1}-{i*4+4}' for i in range(n_folds)]
    x           = np.arange(n_folds)
    bar_w       = 0.28
    n_models    = len(MODEL_NAMES_CLF)
    letters     = [char for char in 'abcdefgh']

    for sensor_name in SENSOR_NAMES_CLF:
        slabel = sensor_short_label(sensor_name)
        sshort = sensor_name.split('\u2014')[0].strip().lower().replace(' ', '_')

        for metric, ylabel in [('accuracy', 'Accuracy'), ('f1', 'Macro F1')]:
            fig, ax = plt.subplots()

            for mi, model_name in enumerate(MODEL_NAMES_CLF):
                fm   = loco_results[sensor_name][model_name]['fold_metrics']
                vals = [d[metric] for d in fm]
                offs = x + (mi - (n_models - 1) / 2) * bar_w
                ax.bar(offs, vals, width=bar_w * 0.9,
                       color=COLORS[model_name],
                       edgecolor='black', linewidth=0.5, zorder=3,
                       label=SHORT[model_name])

            ax.axhline(0.95, ls='--', lw=0.6, color='0.3', zorder=2, label='0.95')
            ax.set_xticks(x)
            ax.set_xticklabels(fold_labels, fontsize=6)
            ax.set_ylim(0.0, 1.1)
            ax.set_ylabel(ylabel)
            ax.legend(title='Model', loc='lower right', frameon=True, ncol=1)
            ax.set_title(slabel)

            # ax.text(-0.20, 1.05, f'({letters.pop(0)})', transform=ax.transAxes,
            #         fontsize=8, fontweight='bold', va='top')

            metric_stub = 'acc' if metric == 'accuracy' else 'f1'
            fname = f'{OUT_DIR}/fig_loco_fold_{metric_stub}_{sshort}.pdf'
            plt.savefig(fname)
            print(f'Saved {fname}')
            plt.show()

## Classification (LOCO): Confusion Matrices

Row-normalised, one figure per sensor; columns are models.

In [ ]:
if loco_available:
    class_names = le.classes_

    for sensor_name in SENSOR_NAMES_CLF:
        slabel = sensor_short_label(sensor_name)
        fig, axes = plt.subplots(1, len(MODEL_NAMES_CLF),
                                 figsize=figsize('double', 0.4))
        fig.subplots_adjust(left=0.06, right=0.97, bottom=0.14,
                            top=0.88, wspace=0.35)
        fig.suptitle(f'{slabel}: confusion matrices (LOCO, row-normalised)')

        for ax, model_name, plabel in zip(axes, MODEL_NAMES_CLF,
                                           ['(a)', '(b)', '(c)']):
            res     = loco_results[sensor_name][model_name]
            cm      = res['confusion_matrix'].astype(float)
            cm_norm = cm / cm.sum(axis=1, keepdims=True)
            sns.heatmap(cm_norm, annot=False, cmap='Blues', ax=ax,
                        xticklabels=class_names, yticklabels=class_names,
                        linewidths=0.2, #cbar=(ax is axes[-1]),
                        cbar=None,
                        vmin=0, vmax=1, square=True)
            ax.set_title(f'{SHORT[model_name]}  (Acc = {res["accuracy"]:.3f})')
            ax.set_xlabel('Predicted')
            ax.set_ylabel('True' if ax is axes[0] else '')
            ax.tick_params(axis='x', rotation=45, labelsize=6)
            ax.tick_params(axis='y', rotation=0, labelsize=6)
            # panel_label(ax, plabel)
            
        # Extract the mappable from the last heatmap and add it to the figure
        mappable = ax.collections[0]
        fig.colorbar(mappable, ax=axes, shrink=0.8)

        sshort = 'sensor_a' if 'Sensor A' in sensor_name else 'sensor_b'
        fname  = f'{OUT_DIR}/fig_loco_cm_{sshort}.pdf'
        plt.savefig(fname)
        print(f'Saved {fname}')
        plt.show()


## LOCO vs. Stratified K-Fold: Comparison Table

In [ ]:
if loco_available and clf_available:
    rows = []
    for sensor_name in SENSOR_NAMES_CLF:
        sl = sensor_short_label(sensor_name)
        for model_name in MODEL_NAMES_CLF:
            orig  = clf_results[sensor_name][model_name]
            loco  = loco_results[sensor_name][model_name]
            rows.append({
                'Sensor / Model': f'{sl} / {SHORT[model_name]}',
                'KF Acc':   orig['accuracy'],
                'KF F1':    orig['f1'],
                'KF AUC':   orig['roc_auc'],
                'LOCO Acc': loco['accuracy'],
                'LOCO F1':  loco['f1'],
                'LOCO AUC': loco['roc_auc'],
                'Delta Acc': loco['accuracy'] - orig['accuracy'],
            })
    df_compare = pd.DataFrame(rows)
    with pd.option_context('display.float_format', '{:.4f}'.format,
                           'display.width', 120):
        print(df_compare.to_string(index=False))

    df_compare.to_csv(f'{OUT_DIR}/loco_vs_kfold.csv', index=False,
                      float_format='%.4f')
    print(f'\nSaved {OUT_DIR}/loco_vs_kfold.csv')
